# Build `places.geojson` from iNaturalist observations

This notebook reads `data/obs_H_M_L.csv` (iNaturalist observations of
invasive species in Montréal), classifies each row by risk tier based on
its `taxon_id`, filters the photo URL by license (only CC0 / CC-BY /
CC-BY-NC are safe to show), and writes `data/places.geojson` for the
Leaflet page to load.

The last cell is a **check cell**: it re-reads the written file and
verifies the feature count, the geographic bounds, and the presence of
key properties. It must pass before the page is expected to work.

In [1]:
import json, math
from pathlib import Path
import pandas as pd

## 1. Paths

`CSV_PATH` is relative to this notebook: `notebooks/` → up one → `data/`.

In [2]:
CSV_PATH = Path("../data/obs_H_M_L.csv")
OUT_PATH = Path("../data/places.geojson")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Reading:", CSV_PATH.resolve())
print("Writing:", OUT_PATH.resolve())

Reading: C:\Users\Lukas\Documents\GEOG464\observations-793731.csv\data\obs_H_M_L.csv
Writing: C:\Users\Lukas\Documents\GEOG464\observations-793731.csv\data\places.geojson


## 2. Small helpers

`clean` strips whitespace and turns NaN into an empty string; `to_float`
returns `None` for unparseable coordinates; `to_bool` reads iNat's
`"true"` / `"false"` strings.

In [3]:
def clean(v):
    if v is None:
        return ""
    if isinstance(v, float) and math.isnan(v):
        return ""
    return str(v).strip()


def to_float(s):
    s = clean(s)
    if s == "":
        return None
    try:
        f = float(s)
        return None if math.isnan(f) else f
    except ValueError:
        return None


def to_bool(s):
    return clean(s).lower() == "true"

## 3. Risk tiers and taxon info

Three sets of `taxon_id`s, one per risk tier, plus a `TAXON_INFO` dict
that records each taxon's Québec presence status and whether REFEE bans
its cultivation and sale. The subspecies IDs (`5903`, `1611981`,
`487107`, `707386`, `462124`) are included so observations identified
below species level still fall into the right tier.

In [4]:
HIGH_RISK_TAXON_IDS = {
    "56061", "169900", "54763", "78157", "55972", "54811",
    "850784", "593040", "914922", "556361", "60467",
    "5903", "1611981",   # Rhamnus cathartica cathartica (both IDs seen)
}

MODERATE_RISK_TAXON_IDS = {
    "63337", "130228", "75921", "64540", "82342", "632667",
    "208963", "51741", "1555600", "58727", "55850", "77468",
    "81600", "47892", "116710", "60202", "361265", "64697",
    "129038", "165632", "78707", "204321",
    "487107",             # Petasites japonicus giganteus
}

LOW_RISK_TAXON_IDS = {
    "57278", "124544", "966205", "163682", "60462", "59773",
    "77836", "170851", "55724", "123638", "78237", "77206",
    "163336", "47779", "962637", "47697", "62671", "78589",
    "204339", "59549", "165473", "165474", "79463", "55844",
    "56088", "167957", "78882", "83653", "61321", "72340",
    "122970", "56160", "50278",
    "707386", "462124",   # Lamium galeobdolon subspecies
}


def normalize_taxon_id(v):
    s = clean(v).strip('"').strip("'")
    if s.lower() in ("nan", "none", "null"):
        return ""
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def risk_for(taxon_id):
    tid = normalize_taxon_id(taxon_id)
    if tid in HIGH_RISK_TAXON_IDS:     return "high"
    if tid in MODERATE_RISK_TAXON_IDS: return "moderate"
    if tid in LOW_RISK_TAXON_IDS:      return "low"
    return "unknown"

## 4. Taxon info (Québec presence + REFEE ban)

Each entry is `(presence_in_Quebec, banned_under_REFEE)`. Used to fill
`presence` and `banned` properties on each feature so the popup can
display Québec status.

In [5]:
TAXON_INFO = {
    # HIGH RISK
    "56061":  ("Established", True),  "169900": ("Established", True),
    "54763":  ("Established", False), "78157":  ("Established", True),
    "55972":  ("Established", True),  "54811":  ("Established", True),
    "1611981":("Established", True),  "850784": ("Established", True),
    "593040": ("Established", True),  "914922": ("Established", True),
    "556361": ("Established", True),  "60467":  ("Established", True),
    "5903":   ("Established", True),

    # MODERATE RISK
    "63337":  ("Established", False), "130228": ("Established", True),
    "75921":  ("Absent", True),       "64540":  ("Established", True),
    "82342":  ("Absent", True),       "632667": ("Established", True),
    "208963": ("Established", True),  "51741":  ("Established", True),
    "1555600":("Absent", True),       "58727":  ("Established", True),
    "55850":  ("Established", False), "77468":  ("Absent", True),
    "81600":  ("Established", True),  "47892":  ("Established", True),
    "116710": ("Absent", True),       "60202":  ("Absent", True),
    "361265": ("Established", False), "64697":  ("Absent", True),
    "129038": ("Established", False), "165632": ("Established", True),
    "78707":  ("Established", True),  "204321": ("Established", True),
    "487107": ("Established", False),

    # LOW RISK
    "57278":  ("Established", False), "124544": ("Established", False),
    "966205": ("Established", False), "163682": ("Established", True),
    "60462":  ("Established", False), "59773":  ("Established", False),
    "77836":  ("Established", False), "170851": ("Established", False),
    "55724":  ("Established", False), "123638": ("Established", False),
    "78237":  ("Established", True),  "77206":  ("Established", False),
    "163336": ("Established", False), "47779":  ("Established", False),
    "962637": ("Single observation", False),
    "47697":  ("Established", False), "62671":  ("Absent", True),
    "78589":  ("Single observation", False),
    "204339": ("Established", False), "59549":  ("Established", False),
    "165473": ("Established", False), "165474": ("Established", False),
    "79463":  ("Established", False), "55844":  ("Established", False),
    "56088":  ("Established", False), "167957": ("Established", False),
    "78882":  ("Established", False), "83653":  ("Established", False),
    "61321":  ("Established", False), "72340":  ("Single observation", False),
    "122970": ("Established", False), "56160":  ("Established", False),
    "50278":  ("Established", False),
    "707386": ("Established", False), "462124": ("Established", False),
}

## 5. License filtering

Every observation is kept (facts are not copyrightable), but the photo
URL is only carried into the output when the license permits reuse:

- `CC0`, `CC-BY`, `CC-BY-NC` → show the photo
- `CC-BY-SA`, `CC-BY-NC-SA`, `CC-BY-ND`, `CC-BY-NC-ND` → drop the photo
- blank (All Rights Reserved) → drop the photo

In [6]:
PHOTO_OK_LICENSES = {"CC0", "CC-BY", "CC-BY-NC"}


def normalize_license(v):
    s = clean(v).upper()
    s = s.replace(" ", "-").replace("_", "-")
    s = s.replace("--", "-").strip("-")
    return s

## 6. Read and clean

Read the CSV, strip BOM/whitespace from column names, then drop rows
with duplicate `id` values (keeps the first occurrence). We count how
many rows were dropped so the number is documented, not hidden.

In [7]:
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
print(f"Read {len(df)} rows from {CSV_PATH}")

before = len(df)
df = df.drop_duplicates(subset="id", keep="first")
dropped_dupes = before - len(df)
print(f"Dropped {dropped_dupes} duplicate ids → {len(df)} unique observations remain")

Read 8062 rows from ..\data\obs_H_M_L.csv
Dropped 0 duplicate ids → 8062 unique observations remain


## 7. Build GeoJSON features

For each row: classify risk from `taxon_id`, look up presence/banned in
`TAXON_INFO`, filter the photo by license, and build a feature with a
point geometry (or `null` geometry when the coords are missing).

In [8]:
features = []
null_geom = 0
obscured_count = 0
geoprivacy_count = 0
counts = {"high": 0, "moderate": 0, "low": 0, "unknown": 0}
photos_dropped_license = 0
license_seen = {}

for _, row in df.iterrows():
    lic_raw = clean(row.get("license"))
    lic = normalize_license(lic_raw)
    license_seen[lic if lic else "(blank)"] = license_seen.get(lic if lic else "(blank)", 0) + 1

    photo_url = clean(row.get("image_url"))
    if photo_url and lic not in PHOTO_OK_LICENSES:
        photo_url = ""
        photos_dropped_license += 1

    lat = to_float(row.get("latitude"))
    lon = to_float(row.get("longitude"))
    if lat is None or lon is None:
        geometry = None
        null_geom += 1
    else:
        geometry = {"type": "Point", "coordinates": [lon, lat]}

    obscured = to_bool(row.get("coordinates_obscured"))
    if obscured:
        obscured_count += 1

    taxon_geoprivacy = clean(row.get("taxon_geoprivacy"))
    if taxon_geoprivacy:
        geoprivacy_count += 1

    taxon_id = clean(row.get("taxon_id"))
    risk = risk_for(taxon_id)
    counts[risk] += 1

    presence, banned = TAXON_INFO.get(taxon_id, ("", False))

    features.append({
        "type": "Feature",
        "geometry": geometry,
        "properties": {
            "id": clean(row.get("id")),
            "observed_on": clean(row.get("observed_on")),
            "time_observed_at": clean(row.get("time_observed_at")),
            "user_login": clean(row.get("user_login")),
            "license": lic,
            "url": clean(row.get("url")),
            "image_url": photo_url,
            "place_guess": clean(row.get("place_guess")),
            "place_town_name": clean(row.get("place_town_name")),
            "species_guess": clean(row.get("species_guess")),
            "scientific_name": clean(row.get("scientific_name")),
            "common_name": clean(row.get("common_name")),
            "iconic_taxon_name": clean(row.get("iconic_taxon_name")),
            "taxon_id": taxon_id,
            "positional_accuracy": clean(row.get("positional_accuracy")),
            "public_positional_accuracy": clean(row.get("public_positional_accuracy")),
            "coordinates_obscured": obscured,
            "taxon_geoprivacy": taxon_geoprivacy,
            "risk": risk,
            "presence": presence,
            "banned": banned,
        },
    })

print(f"Built {len(features)} features")

Built 8062 features


## 8. Write the GeoJSON

In [9]:
geojson = {"type": "FeatureCollection", "features": features}

tmp = OUT_PATH.with_suffix(OUT_PATH.suffix + ".tmp")
with open(tmp, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)
tmp.replace(OUT_PATH)

print("\nLicenses found in CSV:")
for k, v in sorted(license_seen.items(), key=lambda kv: -kv[1]):
    print(f"  {k:>12}: {v}")

print(
    f"\nWrote {len(features)} features "
    f"(high={counts['high']}, moderate={counts['moderate']}, "
    f"low={counts['low']}, unknown={counts['unknown']}; "
    f"photos dropped for license: {photos_dropped_license}; "
    f"null_geom={null_geom}, {obscured_count} coordinate-obscured, "
    f"{geoprivacy_count} with taxon_geoprivacy) → {OUT_PATH}"
)


Licenses found in CSV:
      CC-BY-NC: 5570
       (blank): 1361
         CC-BY: 728
           CC0: 174
   CC-BY-NC-ND: 143
      CC-BY-SA: 52
   CC-BY-NC-SA: 33
      CC-BY-ND: 1

Wrote 8062 features (high=3724, moderate=725, low=3613, unknown=0; photos dropped for license: 1590; null_geom=0, 0 coordinate-obscured, 5833 with taxon_geoprivacy) → ..\data\places.geojson


## 9. Check cell

Re-read the written file and test four things:

1. Every cleaned row became a feature (no silent drops).
2. Every feature with a geometry sits inside the Montreal bounding box.
3. Every feature has the required properties populated.
4. The risk tiers sum to the total feature count.

If all four print `✓`, the pipeline worked.

In [11]:
gj = json.loads(OUT_PATH.read_text(encoding="utf-8"))
feats = gj["features"]

# 1. count
assert len(feats) == len(df), (
    f"Count mismatch: {len(feats)} features vs {len(df)} cleaned rows"
)
print(f"✓ {len(feats)} features matches {len(df)} cleaned rows")

# 2. bounds — Montreal agglomeration roughly lat 45.30–45.75, lon -74.10–-73.30
LAT_MIN, LAT_MAX = 45.30, 45.75
LON_MIN, LON_MAX = -74.10, -73.30
out_of_bounds = 0
for f in feats:
    if f["geometry"] is None:
        continue
    lon, lat = f["geometry"]["coordinates"]
    if not (LAT_MIN <= lat <= LAT_MAX and LON_MIN <= lon <= LON_MAX):
        out_of_bounds += 1
assert out_of_bounds == 0, f"{out_of_bounds} features outside the Montreal box"
print(f"✓ All geometries inside Montreal bounds")

# 3. required properties
REQUIRED = ["id", "risk", "taxon_id", "scientific_name"]
missing = 0
for f in feats:
    for key in REQUIRED:
        if not f["properties"].get(key):
            missing += 1
assert missing == 0, f"{missing} features missing required properties"
print(f"✓ All features have {REQUIRED}")

# 4. tier sum
tier_sum = sum(counts.values())
assert tier_sum == len(feats), f"Tier sum {tier_sum} ≠ features {len(feats)}"
print(f"✓ Risk tiers sum to {tier_sum}: "
      f"high={counts['high']}, moderate={counts['moderate']}, "
      f"low={counts['low']}, unknown={counts['unknown']}")

print("\nAll checks passed.")

✓ 8062 features matches 8062 cleaned rows
✓ All geometries inside Montreal bounds
✓ All features have ['id', 'risk', 'taxon_id', 'scientific_name']
✓ Risk tiers sum to 8062: high=3724, moderate=725, low=3613, unknown=0

All checks passed.
